# 00 — Instance setup and recovery

## Goal
Prepare the **AHR Go foundation**, not a trained playing model. This notebook checks
the current hardware, records the environment, and tests interrupted/resumed work
in separate fresh Python kernels on the CPU and **each GPU**.

**Inputs:** this project, its `.venv` Python kernel, notebook packages and PyTorch.
**Outputs:** `config/project.json`, `reports/00_environment.json`,
`reports/00_recovery_validation.json`, package locks and small smoke checkpoints.
**Expected runtime:** about 1–3 minutes after dependencies are installed; no AHR training.
**Checkpoint location:** `checkpoints/setup-smoke-v1-<device>/`.
**Exact resume:** restart the kernel, choose **AHR Go (.venv)**, and Run All.
The smoke resumes from the latest valid checkpoint; completed runs are checked again
without advancing the saved run. The uninterrupted reference is recomputed for comparison.
Changing the smoke configuration requires a new run ID.

## Boundaries
All authored remote executable project logic stays in `.ipynb` cells. Installed
third-party packages are dependencies, not authored project helper files.
The next three notebooks establish rules, bounded constraints, and small initial
datasets. They cannot establish near-perfect Go strength.

## Setup — project and dependency contract
Open this notebook under `ahr_go/notebooks/`, or set `AHR_GO_PROJECT` to the project
directory. The kernel must be the project's `.venv`; do not use another project's
runtime. On a fresh machine, the following are **one-time administration commands**
(not a helper script):

```text
python3 -m venv /home/benjamin/workspace/ahr_go/.venv
/home/benjamin/workspace/ahr_go/.venv/bin/python -m pip install torch numpy matplotlib nbformat nbclient nbconvert ipykernel
/home/benjamin/workspace/ahr_go/.venv/bin/python -m ipykernel install --user --name ahr-go --display-name "AHR Go (.venv)"
```

After this delivery, prefer `config/requirements-lock.txt` over unpinned packages.
Dependencies are already installed on the current instance. Run All does not change
drivers, install packages, mount disks, expose a service, or alter other projects.

In [ ]:
from pathlib import Path
import os, json, sys, platform, subprocess, shutil, hashlib, time
import importlib.metadata
import nbformat
from nbclient import NotebookClient

candidate = Path(os.environ.get("AHR_GO_PROJECT", Path.cwd())).resolve()
PROJECT_ROOT = candidate.parent if candidate.name == "notebooks" else candidate
assert (PROJECT_ROOT / "notebooks" / "00_instance_setup_and_recovery.ipynb").is_file(), "Open the project notebook directory or set AHR_GO_PROJECT."
assert PROJECT_ROOT.name == "ahr_go" and PROJECT_ROOT != Path("/root/ahr_go"), "Use workspace/ahr_go."
for folder in ("config", "reports", "checkpoints", "data", "logs"):
    (PROJECT_ROOT / folder).mkdir(exist_ok=True)
os.environ["AHR_GO_PROJECT"] = str(PROJECT_ROOT)
EXPECTED_KERNEL = PROJECT_ROOT / ".venv"
assert Path(sys.prefix).resolve() == EXPECTED_KERNEL.resolve(), f"Select AHR Go (.venv), not {sys.prefix}"
print(f"Project: {PROJECT_ROOT}\nPython: {sys.version.split()[0]}\nKernel: {sys.prefix}")

## Reusable definitions — safe notebook-only loading and atomic artifacts
Only cells tagged `ahr-export` are reusable definitions. Loading a notebook must
never execute its validation, dataset generation, or future training workflow.
These exports are trusted project code: do not load arbitrary downloaded notebooks.

In [ ]:
from pathlib import Path
import os, json, hashlib, tempfile, warnings, time, copy, random, uuid, fcntl
from contextlib import contextmanager
import numpy as np
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch

def atomic_write_bytes(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=path.parent, prefix="." + path.name, suffix=".tmp", delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(content)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
        directory_fd = os.open(path.parent, os.O_RDONLY)
        try:
            os.fsync(directory_fd)
        finally:
            os.close(directory_fd)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def atomic_json(path, value):
    atomic_write_bytes(path, (json.dumps(value, indent=2, sort_keys=True) + "\n").encode())

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_notebook_exports(path, namespace):
    import nbformat
    notebook = nbformat.read(path, as_version=4)
    selected = [cell for cell in notebook.cells if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", [])]
    if not selected:
        raise ValueError(f"No explicitly tagged definitions: {path}")
    for index, cell in enumerate(selected):
        exec(compile(cell.source, f"{Path(path).name}:export-{index}", "exec"), namespace)
    return len(selected)

REQUIRED_CHECKPOINT_KEYS = {"model", "optimizer", "scheduler", "scaler", "rng", "sampler", "global_step", "config", "metrics"}

def checkpoint_due(global_step, last_step, last_time, stage_boundary=False):
    return stage_boundary or global_step - last_step >= 250 or time.monotonic() - last_time >= 900

def _checkpoint_entry_state(run_dir, entry):
    if Path(entry["file"]).name != entry["file"]:
        raise ValueError("Checkpoint manifest must use simple filenames")
    path = Path(run_dir) / entry["file"]
    if not path.is_file() or file_sha256(path) != entry["sha256"]:
        return None
    try:
        state = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(state, dict) or REQUIRED_CHECKPOINT_KEYS - set(state):
            return None
        if type(state["global_step"]) is not int or state["global_step"] != entry["global_step"]:
            return None
        return state
    except Exception:
        return None

@contextmanager
def checkpoint_lock(run_dir):
    directory = Path(run_dir)
    directory.mkdir(parents=True, exist_ok=True)
    with open(directory / ".checkpoint.lock", "a+b") as handle:
        try:
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError(f"Another process owns this run: {directory}") from error
        try:
            yield
        finally:
            fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

def _save_checkpoint_unlocked(run_dir, state, *, best=False, resume_cell="00: Recovery smoke — Run All"):
    """Single-writer, interruption-safe manifest commit; retain latest two + best."""
    import io
    missing = REQUIRED_CHECKPOINT_KEYS - set(state)
    if missing:
        raise ValueError(f"Incomplete resume state: {sorted(missing)}")
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = run_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {"schema_version": 1, "entries": [], "best": None}
    step = int(state["global_step"])
    if step < 0:
        raise ValueError("Negative global step")
    valid_entries = []
    for entry in manifest["entries"]:
        if _checkpoint_entry_state(run_dir, entry) is not None:
            valid_entries.append(entry)
    if valid_entries and step <= max(entry["global_step"] for entry in valid_entries):
        raise ValueError("Refusing to overwrite or rewind committed steps; resume or use a new run ID.")
    # Invalid newer payloads stay on disk for inspection but are not promoted again.
    manifest["entries"] = valid_entries
    if manifest.get("best") not in {entry["file"] for entry in valid_entries}:
        manifest["best"] = None
    payload = io.BytesIO()
    torch.save(state, payload)
    filename = f"step-{step:09d}-{uuid.uuid4().hex[:8]}.pt"
    atomic_write_bytes(run_dir / filename, payload.getvalue())
    entry = {"file": filename, "sha256": file_sha256(run_dir / filename), "global_step": step}
    entries = sorted(manifest["entries"] + [entry], key=lambda item: item["global_step"])
    best_file = filename if best or manifest.get("best") is None else manifest.get("best")
    retained = {item["file"] for item in entries[-2:]}
    if best_file:
        retained.add(best_file)
    manifest.update(entries=[item for item in entries if item["file"] in retained], latest=filename,
                    best=best_file, resume_cell=resume_cell, checkpoint_policy="every stage/epoch and <=15 minutes or 250 optimizer steps, whichever first")
    atomic_json(manifest_path, manifest)
    # Prune only superseded checkpoint files identified in the prior manifest.
    for item in entries:
        if item["file"] not in retained:
            (run_dir / item["file"]).unlink(missing_ok=True)
    return entry

def _load_latest_checkpoint_unlocked(run_dir):
    """Load only hash-verified, self-authored checkpoints. Never use untrusted .pt files."""
    run_dir = Path(run_dir)
    manifest_path = run_dir / "manifest.json"
    if not manifest_path.exists():
        if list(run_dir.glob("*.pt")):
            raise RuntimeError("Checkpoint files exist but their committed manifest is missing; inspect rather than restart.")
        return None
    manifest = json.loads(manifest_path.read_text())
    failed = []
    for entry in sorted(manifest["entries"], key=lambda item: item["global_step"], reverse=True):
        state = _checkpoint_entry_state(run_dir, entry)
        if state is None:
            failed.append(entry["file"])
            continue
        if failed:
            warnings.warn(f"Rejected invalid newer checkpoint(s); recovering from step {state['global_step']}: {failed}")
        return state
    raise RuntimeError(f"No valid committed checkpoint. Refusing silent restart: {failed}")

def save_checkpoint(run_dir, state, *, best=False, resume_cell="00: Recovery smoke — Run All"):
    with checkpoint_lock(run_dir):
        return _save_checkpoint_unlocked(run_dir, state, best=best, resume_cell=resume_cell)

def load_latest_checkpoint(run_dir):
    with checkpoint_lock(run_dir):
        return _load_latest_checkpoint_unlocked(run_dir)

def capture_rng():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch_cpu": torch.get_rng_state(),
            "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    if state["torch_cuda"]:
        torch.cuda.set_rng_state_all(state["torch_cuda"])

## Hardware, storage, rules and environment receipt
The storage check records the actual filesystem device and Linux disk identity.
A **persistent boot disk** survives an ordinary reboot/stop, but is not an independent
backup: its deletion policy may remove it with the VM. Local NVMe scratch is not
persistent. Never recycle/destroy this instance before verifying an off-instance
copy of notebooks, manifests, datasets and final/best checkpoints.

The initial Go rules use a 9×9 board, 7.5 komi, area scoring, no suicide and positional
superko. Dead stones must be played out/captured; we do not invent a dead-stone adjudicator.

In [ ]:
def command_output(arguments):
    result = subprocess.run(arguments, capture_output=True, text=True, timeout=20)
    return {"returncode": result.returncode, "stdout": result.stdout.strip(), "stderr": result.stderr.strip()}

gpu = command_output(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"])
assert gpu["returncode"] == 0, gpu["stderr"]
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "Two working CUDA GPUs are required on this instance."
gpu_checks = []
for index in range(torch.cuda.device_count()):
    a = torch.arange(16, device=f"cuda:{index}", dtype=torch.float32).reshape(4, 4)
    assert torch.equal((a @ torch.eye(4, device=a.device)).cpu(), a.cpu())
    torch.cuda.synchronize(index)
    prop = torch.cuda.get_device_properties(index)
    gpu_checks.append({"device": index, "name": prop.name, "memory_gib": round(prop.total_memory / 2**30, 2), "matrix_check": "PASS"})
mount = command_output(["findmnt", "-J", "-T", str(PROJECT_ROOT), "-o", "TARGET,SOURCE,FSTYPE,OPTIONS"])
disk_ids = sorted(path.name for path in Path("/dev/disk/by-id").glob("*"))
source = json.loads(mount["stdout"])["filesystems"][0]["source"] if mount["returncode"] == 0 else "unknown"
matching_ids = [path.name for path in Path("/dev/disk/by-id").glob("*") if path.resolve() == Path(source).resolve()]
persistent = any("PersistentDisk" in name for name in matching_ids)
storage = {"mount": mount, "matching_disk_ids": matching_ids, "persistent_volume_identified": persistent,
           "instance_delete_policy": "NOT VERIFIED: backup required before deletion/recycle",
           "free_gib": round(shutil.disk_usage(PROJECT_ROOT).free / 2**30, 2)}
if not persistent:
    warnings.warn("Persistent volume not established. Recycle/destroy may erase the workspace; off-instance backups are required.")
packages = {name: importlib.metadata.version(name) for name in ("torch", "numpy", "matplotlib", "nbformat", "nbclient", "nbconvert", "ipykernel")}
environment = {"schema_version": 1, "host": platform.node(), "project": str(PROJECT_ROOT), "python": sys.version,
               "executable": sys.executable, "cpu_threads": os.cpu_count(), "gpu": gpu, "gpu_checks": gpu_checks,
               "torch_cuda": torch.version.cuda, "packages": packages, "storage": storage, "checked_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
project_config = {"schema_version": 1, "project": "ahr_go", "board_size": 9, "komi": 7.5,
                  "scoring": "area; dead stones must be captured", "ko": "positional_superko; pass exempt",
                  "suicide": False, "termination": "two consecutive passes", "first_notebooks": ["00", "01", "02", "03"],
                  "seed": 20260923, "checkpoint_minutes": 15, "checkpoint_optimizer_steps": 250,
                  "checkpoint_keep": "latest two and best", "no_model_training_in_foundation": True}
config_path = PROJECT_ROOT / "config" / "project.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == project_config, "Existing project config differs. Review explicitly instead of overwriting."
else:
    atomic_json(config_path, project_config)
atomic_json(PROJECT_ROOT / "reports" / "00_environment.json", environment)
lock = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
atomic_write_bytes(PROJECT_ROOT / "config" / "requirements-lock.txt", lock.encode())
print(json.dumps({"GPUs": gpu_checks, "CPU_threads": os.cpu_count(), "storage": {key: storage[key] for key in ("persistent_volume_identified", "free_gib", "instance_delete_policy")}, "packages": packages}, indent=2))

## Recovery definitions — tiny deterministic training fixture
This is an **8-step synthetic recovery fixture**, not the AHR Go model. It exercises
model/optimizer/scheduler/scaler state, Python/NumPy/CPU/GPU random generators, the
sampler cursor, metrics and configuration. First execution saves step 3 in one kernel;
a different kernel resumes to step 8 and compares every state field with uninterrupted
execution. Completed reruns perform no new checkpointed updates.

Exact bitwise continuation is tested on the same hardware/software. It is not a
promise of bitwise equality across different driver, library or GPU versions.

In [ ]:
def setup_smoke_components(device):
    random.seed(20260923)
    np.random.seed(20260923)
    torch.manual_seed(20260923)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(20260923)
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(1)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
    model = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Dropout(0.2), torch.nn.Linear(8, 1)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.9)
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    sampler = {"permutation": list(range(64)), "position": 0, "epoch": 0}
    random.shuffle(sampler["permutation"])
    return model, optimizer, scheduler, scaler, sampler

def smoke_advance(components, device, start, stop, losses):
    model, optimizer, scheduler, scaler, sampler = components
    model.train()
    for step in range(start, stop):
        x = torch.randn(8, 4).to(device)
        y = x.sum(dim=1, keepdim=True) * (0.4 + random.random() * 0.1) + float(np.random.normal(0, 0.05))
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda" if str(device).startswith("cuda") else "cpu", dtype=torch.float16, enabled=str(device).startswith("cuda")):
            loss = ((model(x) - y) ** 2).mean()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(float(loss.detach().cpu()))
        sampler["position"] += 8
        if sampler["position"] >= len(sampler["permutation"]):
            sampler["position"] = 0
            sampler["epoch"] += 1
            random.shuffle(sampler["permutation"])
    return losses

def smoke_state(components, device, step, losses):
    model, optimizer, scheduler, scaler, sampler = components
    return copy.deepcopy({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                          "scaler": scaler.state_dict(), "rng": capture_rng(), "sampler": sampler, "global_step": step,
                          "config": {"fixture": "setup-smoke-v1", "device": device, "seed": 20260923, "total_steps": 8}, "metrics": {"losses": losses}})

def restore_smoke(state, device):
    components = setup_smoke_components(device)
    model, optimizer, scheduler, scaler, sampler = components
    assert state["config"] == {"fixture": "setup-smoke-v1", "device": device, "seed": 20260923, "total_steps": 8}
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    sampler.update(state["sampler"])
    restore_rng(state["rng"])
    return components

def nested_equal(left, right):
    if isinstance(left, torch.Tensor):
        return isinstance(right, torch.Tensor) and torch.equal(left.cpu(), right.cpu())
    if isinstance(left, np.ndarray):
        return isinstance(right, np.ndarray) and np.array_equal(left, right)
    if isinstance(left, dict):
        return isinstance(right, dict) and left.keys() == right.keys() and all(nested_equal(left[key], right[key]) for key in left)
    if isinstance(left, (list, tuple)):
        return isinstance(right, (list, tuple)) and len(left) == len(right) and all(nested_equal(a, b) for a, b in zip(left, right))
    return left == right

def smoke_phase(project, phase):
    receipts = []
    for device in ["cpu"] + [f"cuda:{index}" for index in range(torch.cuda.device_count())]:
        run_dir = Path(project) / "checkpoints" / ("setup-smoke-v1-" + device.replace(":", "-"))
        existing = load_latest_checkpoint(run_dir)
        if phase == "interrupt":
            if existing is None:
                components = setup_smoke_components(device)
                losses = smoke_advance(components, device, 0, 3, [])
                save_checkpoint(run_dir, smoke_state(components, device, 3, losses), best=True)
                loaded = load_latest_checkpoint(run_dir)
                assert nested_equal(loaded, smoke_state(components, device, 3, losses))
                receipts.append({"device": device, "saved_step": 3, "action": "interrupted_after_checkpoint"})
            else:
                receipts.append({"device": device, "saved_step": existing["global_step"], "action": "existing_checkpoint_preserved"})
        elif phase == "resume":
            assert existing is not None, "Run the interrupt phase first."
            start = existing["global_step"]
            components = restore_smoke(existing, device)
            losses = list(existing["metrics"]["losses"])
            for stop in (6, 8):
                if start < stop:
                    losses = smoke_advance(components, device, start, stop, losses)
                    state = smoke_state(components, device, stop, losses)
                    save_checkpoint(run_dir, state, best=losses[-1] <= min(losses))
                    start = stop
            resumed = smoke_state(components, device, 8, losses)
            reference_components = setup_smoke_components(device)
            reference_losses = smoke_advance(reference_components, device, 0, 8, [])
            reference = smoke_state(reference_components, device, 8, reference_losses)
            equal = nested_equal(resumed, reference)
            assert equal, f"Resume differs from uninterrupted execution on {device}"
            assert nested_equal(load_latest_checkpoint(run_dir), resumed)
            manifest = json.loads((run_dir / "manifest.json").read_text())
            assert {6, 8}.issubset({entry["global_step"] for entry in manifest["entries"]})
            assert manifest["best"] in {entry["file"] for entry in manifest["entries"]}
            receipts.append({"device": device, "resumed_from_step": existing["global_step"], "final_step": 8,
                             "exact_all_state_match": equal, "final_loss": losses[-1], "retained_checkpoints": len(manifest["entries"]),
                             "new_checkpointed_optimizer_steps": 8 - existing["global_step"]})
        else:
            raise ValueError(phase)
    return receipts

## Recovery smoke — fresh-kernel interruption and resume
The following starts two short **in-memory notebooks** using the same project kernel.
No helper source file is created. Each child kernel exits before the next begins.
Only the four delivered `.ipynb` files contain project source.

In [ ]:
source_notebook = nbformat.read(PROJECT_ROOT / "notebooks" / "00_instance_setup_and_recovery.ipynb", as_version=4)
exports = "\n\n".join(cell.source for cell in source_notebook.cells if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []))
phase_receipts = {}
with checkpoint_lock(PROJECT_ROOT / "checkpoints" / "setup-smoke-owner"):
    for phase in ("interrupt", "resume"):
        child = nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(exports),
            nbformat.v4.new_code_cell(f"import sys\nassert Path(sys.prefix).resolve() == Path({str(PROJECT_ROOT)!r}, '.venv').resolve(), 'Wrong child kernel environment'\nresult = smoke_phase({str(PROJECT_ROOT)!r}, {phase!r})\nfor row in result:\n    row['child_runtime'] = {{'executable': sys.executable, 'torch': torch.__version__, 'cuda': torch.version.cuda}}\nprint(json.dumps(result))")])
        executed = NotebookClient(child, timeout=180, kernel_name="ahr-go", resources={"metadata": {"path": str(PROJECT_ROOT)}}).execute()
        output = "".join(item.get("text", "") for item in executed.cells[-1].outputs if item.output_type == "stream" and item.name == "stdout")
        phase_receipts[phase] = json.loads(output.strip())
        print(phase + ":")
        for row in phase_receipts[phase]:
            if phase == "interrupt":
                print(f"  {row['device']}: checkpoint step {row['saved_step']} ({row['action']})")
            else:
                print(f"  {row['device']}: step {row['resumed_from_step']} -> {row['final_step']}; exact state match={row['exact_all_state_match']}; new saved-run steps={row['new_checkpointed_optimizer_steps']}")
        print("  Verified child kernel: " + phase_receipts[phase][0]["child_runtime"]["executable"])
assert all(item["exact_all_state_match"] for item in phase_receipts["resume"])

## Failure and policy checks
An isolated temporary fixture deliberately corrupts its latest checkpoint and
checks fallback to the previous verified checkpoint. It also confirms incomplete
state and missing/corrupt manifests are not silently treated as a new run.
Temporary test artifacts alone are removed automatically; actual smoke checkpoints stay.

In [ ]:
with tempfile.TemporaryDirectory(prefix="recovery-audit-", dir=PROJECT_ROOT / "reports") as temporary:
    directory = Path(temporary)
    with checkpoint_lock(directory):
        try:
            with checkpoint_lock(directory):
                raise AssertionError("Duplicate run lock was accepted")
        except RuntimeError:
            duplicate_writer_rejected = True
    base = load_latest_checkpoint(PROJECT_ROOT / "checkpoints" / "setup-smoke-v1-cpu")
    for step in (1, 2, 3):
        state = copy.deepcopy(base)
        state["global_step"] = step
        save_checkpoint(directory, state, best=step == 1)
    manifest = json.loads((directory / "manifest.json").read_text())
    assert len(manifest["entries"]) == 3
    atomic_write_bytes(directory / manifest["latest"], b"deliberate-corruption")
    with warnings.catch_warnings(record=True) as caught:
        assert load_latest_checkpoint(directory)["global_step"] == 2
        assert len(caught) == 1
    with warnings.catch_warnings(record=True):
        restored = load_latest_checkpoint(directory)
    restored["global_step"] = 3
    save_checkpoint(directory, restored)
    assert load_latest_checkpoint(directory)["global_step"] == 3
    # A valid hash is insufficient: the payload step must match its manifest.
    import io
    bad_payload = io.BytesIO()
    torch.save(restored, bad_payload)
    bad_file = directory / "deliberate-semantic-mismatch.pt"
    atomic_write_bytes(bad_file, bad_payload.getvalue())
    manifest = json.loads((directory / "manifest.json").read_text())
    manifest["entries"].append({"file": bad_file.name, "sha256": file_sha256(bad_file), "global_step": 4})
    manifest["latest"] = bad_file.name
    atomic_json(directory / "manifest.json", manifest)
    with warnings.catch_warnings(record=True):
        assert load_latest_checkpoint(directory)["global_step"] == 3
    restored["global_step"] = 4
    save_checkpoint(directory, restored)
    assert load_latest_checkpoint(directory)["global_step"] == 4
    try:
        save_checkpoint(directory / "incomplete", {"global_step": 0})
    except ValueError:
        incomplete_rejected = True
    else:
        raise AssertionError("Incomplete resume state was accepted")
    (directory / "manifest.json").unlink()
    try:
        load_latest_checkpoint(directory)
    except RuntimeError:
        missing_manifest_rejected = True
    else:
        raise AssertionError("Missing manifest silently restarted")
now = time.monotonic()
assert checkpoint_due(250, 0, now)
assert checkpoint_due(1, 0, now - 901)
assert checkpoint_due(1, 0, now, stage_boundary=True)
assert not checkpoint_due(1, 0, now)
assert len({item["device"] for item in phase_receipts["resume"]}) == torch.cuda.device_count() + 1
report = {"status": "PASS", "fresh_kernel_phases": 2, "devices": phase_receipts["resume"],
          "duplicate_writer_rejected": duplicate_writer_rejected,
          "semantic_mismatch_fallback_and_resave": "PASS",
          "interrupt_phase": phase_receipts["interrupt"], "corrupt_latest_fallback": "PASS", "incomplete_state_rejected": incomplete_rejected,
          "missing_manifest_rejected": missing_manifest_rejected, "checkpoint_cadence_checks": 4,
          "model_training": "NOT STARTED; synthetic recovery fixture only", "persistent_volume_identified": persistent}
first_receipt = PROJECT_ROOT / "reports" / "00_recovery_first_pass.json"
if not first_receipt.exists():
    atomic_json(first_receipt, report)
atomic_json(PROJECT_ROOT / "reports" / "00_recovery_validation.json", report)
print(json.dumps({"status": "PASS", "exact_resume_devices": len(report["devices"]), "failure_handling": "PASS",
                  "checkpoint_cadence": "PASS", "next": "Run notebook 01, then 02, then 03."}, indent=2))

## Takeaways and next steps
The saved outputs above are the evidence for this execution. A successful run means
the two GPUs are accessible and the **small recovery fixture** resumed exactly on
each device. Future AHR training still needs its own interrupted/resumed smoke test,
including its actual replay buffers, data workers and any distributed state.

Use **01 → 02 → 03** next. Keep the project `AGENTS.md` and checkpoint policy in force.
Do not start a long run until its own recovery test passes. No game strength, trained
AHR checkpoint, public server or human-versus-AHR interface is delivered by these four notebooks.